# 𓂀 Hieroglyphics — Siamese Network Training

**Graduation Project — Computer Science**

Trains a **Siamese Network** with SqueezeNet backbone for few-shot hieroglyphic symbol classification.

| Section | Description |
|---------|-------------|
| 1. Setup | Install dependencies & imports |
| 2. Dataset | Siamese pairs from ImageFolder |
| 3. Architecture | EmbeddingNet + SiameseNet |
| 4. Training | BCE loss, Adam optimizer, 20 epochs |
| 5. Evaluation | Cosine similarity classification |
| 6. Embeddings | Save reference embeddings for inference |

In [2]:
!pip install keras-applications


In [3]:
!pip install torch torchvision


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Dataset
import random
from PIL import Image


In [6]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


In [7]:
# 2. Dataset with pairs for Siamese network
class SiameseDataset(Dataset):
    def __init__(self, imageFolderDataset):
        self.imageFolderDataset = imageFolderDataset
        self.classes = imageFolderDataset.classes
        self.class_to_idx = imageFolderDataset.class_to_idx

        # Organize images by class
        self.class_images = {}
        for idx, (img_path, label) in enumerate(self.imageFolderDataset.imgs):
            if label not in self.class_images:
                self.class_images[label] = []
            self.class_images[label].append(img_path)

    def __getitem__(self, index):
        # Choose a random class for anchor
        label1 = random.choice(list(self.class_images.keys()))
        img1_path = random.choice(self.class_images[label1])
        img1 = Image.open(img1_path).convert('RGB')
        img1 = transform(img1)

        # Decide if pair is positive or negative
        should_get_same_class = random.randint(0,1)

        if should_get_same_class:
            label2 = label1
            img2_path = random.choice(self.class_images[label2])
        else:
            label2 = random.choice([x for x in self.class_images.keys() if x != label1])
            img2_path = random.choice(self.class_images[label2])

        img2 = Image.open(img2_path).convert('RGB')
        img2 = transform(img2)

        label = torch.tensor(int(label1 == label2), dtype=torch.float32)

        return img1, img2, label

    def __len__(self):
        return len(self.imageFolderDataset)



In [9]:
# 3. Load ImageFolder
dataset = datasets.ImageFolder(root='/content/drive/MyDrive/datasets/Dataset_organized/organized_dataset_by_class_1', transform=transform)
siamese_dataset = SiameseDataset(dataset)
dataloader = DataLoader(siamese_dataset, shuffle=True, batch_size=32)



In [10]:
# 4. Define SqueezeNet base model for embeddings
class EmbeddingNet(nn.Module):
    def __init__(self):
        super(EmbeddingNet, self).__init__()
        self.squeezenet = models.squeezenet1_1(pretrained=True)
        self.squeezenet.classifier = nn.Sequential()  # remove classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))

    def forward(self, x):
        x = self.squeezenet.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return x

In [11]:
# 5. Siamese Network
class SiameseNet(nn.Module):
    def __init__(self, embedding_net):
        super(SiameseNet, self).__init__()
        self.embedding_net = embedding_net
        self.fc = nn.Sequential(
            nn.Linear(512, 256), # 512 is output of squeezenet features flattened
            nn.ReLU(inplace=True),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1)
        )

    def forward(self, x1, x2):
        out1 = self.embedding_net(x1)
        out2 = self.embedding_net(x2)
        diff = torch.abs(out1 - out2)
        out = self.fc(diff)
        return torch.sigmoid(out)

In [12]:
# 6. Instantiate model
embedding_net = EmbeddingNet()
model = SiameseNet(embedding_net)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=SqueezeNet1_1_Weights.IMAGENET1K_V1`. You can also use `weights=SqueezeNet1_1_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [13]:
# 7. Loss and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
# 8. Training loop
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (img1, img2, label) in enumerate(dataloader):
        img1, img2, label = img1.to(device), img2.to(device), label.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(img1, img2)
        loss = criterion(outputs, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predicted = (outputs > 0.5).float()
        correct += (predicted == label).sum().item()
        total += label.size(0)

    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss / len(dataloader):.4f}, Accuracy: {accuracy:.2f}%")


Epoch 1/10, Loss: 0.4983, Accuracy: 65.32%
Epoch 2/10, Loss: 0.3977, Accuracy: 87.74%
Epoch 3/10, Loss: 0.3566, Accuracy: 90.34%
Epoch 4/10, Loss: 0.3368, Accuracy: 90.92%
Epoch 5/10, Loss: 0.2953, Accuracy: 92.17%
Epoch 6/10, Loss: 0.2633, Accuracy: 93.79%
Epoch 7/10, Loss: 0.2641, Accuracy: 93.15%
Epoch 8/10, Loss: 0.2207, Accuracy: 94.16%
Epoch 9/10, Loss: 0.1969, Accuracy: 94.68%
Epoch 10/10, Loss: 0.1912, Accuracy: 95.54%


In [ ]:
#Test The Model
from PIL import Image
import torch
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

img1 = Image.open("/content/drive/MyDrive/datasets/Dataset_organized/organized_dataset_by_class_1/D1/030114_D1.png").convert('RGB')
img2 = Image.open("/content/drive/MyDrive/datasets/Dataset_organized/organized_dataset_by_class_1/Aa28/050295_Aa28.png").convert('RGB')

img1 = transform(img1).unsqueeze(0).to(device)
img2 = transform(img2).unsqueeze(0).to(device)

model.eval()

with torch.no_grad():
    output = model(img1, img2)
    similarity_score = output.item()

print(f"Similarity Score: {similarity_score:.4f}")
if similarity_score > 0.5:
    print("Images are similar")
else:
    print("Images are Dissimilar")


Similarity Score: 0.0000
Images are Dissimilar


In [19]:
torch.save(model.state_dict(), 'siamese_squeezenet.pth')
print("Model saved successfully!")


Model saved successfully!


In [24]:
import torch
import os
import pickle
from tqdm import tqdm

def save_embeddings(embedding_net, dataset, save_path, device):
    embedding_net.eval()
    embeddings = []

    with torch.no_grad():
        for img, label in tqdm(dataset, desc="Extracting embeddings"):
            img_tensor = img.unsqueeze(0).to(device)
            emb = embedding_net(img_tensor).cpu().squeeze().numpy()
            embeddings.append({
                'embedding': emb,
                'label': label,
                'class_name': dataset.classes[label]
            })

    with open(save_path, 'wb') as f:
        pickle.dump(embeddings, f)

    print(f"Saved {len(embeddings)} embeddings to {save_path}")


In [25]:
save_embeddings(embedding_net, reference_dataset, "saved_embeddings.pkl", device)


Extracting embeddings: 100%|██████████| 3270/3270 [02:18<00:00, 23.57it/s]

Saved 3270 embeddings to saved_embeddings.pkl


In [26]:
import numpy as np
from scipy.spatial.distance import cosine

def classify_using_embeddings(input_image_path, embedding_net, embeddings_file, transform, device):
    # 1. Load saved embeddings
    with open(embeddings_file, 'rb') as f:
        embeddings_data = pickle.load(f)

    # 2. Prepare input image
    image = Image.open(input_image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)

    # 3. Get embedding for input image
    embedding_net.eval()
    with torch.no_grad():
        query_emb = embedding_net(image_tensor).cpu().squeeze().numpy()

    # 4. Find most similar embedding
    best_score = float('inf')
    predicted_class = None

    for item in embeddings_data:
        ref_emb = item['embedding']
        score = cosine(query_emb, ref_emb)  # smaller = more similar
        if score < best_score:
            best_score = score
            predicted_class = item['class_name']

    print(f"Predicted class: {predicted_class} (similarity = {1 - best_score:.4f})")
    return predicted_class


In [27]:
classify_using_embeddings("/content/drive/MyDrive/datasets/Dataset_organized/organized_dataset_by_class_1/D1/030114_D1.png", embedding_net, "saved_embeddings.pkl", transform, device)


Predicted class: D1 (similarity = 1.0000)


'D1'

In [32]:
from torchvision import datasets, transforms
import pickle

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

dataset_path = "/content/drive/MyDrive/datasets/Dataset_organized/organized_dataset_by_class_1"
reference_dataset = datasets.ImageFolder(root=dataset_path, transform=transform)

# استخراج الـ labels كـ أسماء (strings)
reference_labels = [reference_dataset.classes[label] for _, label in reference_dataset]

# تخزينهم
with open("reference_labels.pkl", "wb") as f:
    pickle.dump(reference_labels, f)
